# Image-Level EDA
The bias matching process applied in the previous metadata EDA notebook tells us that the bias matching can be applied to reduce the bias in the dataset. However, it is also important to perform an image-level EDA to understand the distribution of images and their characteristics.

Crucially, this notebook combines multiple per source manifests in `data/interim` (produced by `scripts/build_manifest.py`) into a single combined manifest (`data/interim/manifest_final.parquet`) with paths pointing to the final, model-ready image folders under `data/processed/`, allowing for an overall image analysis. Train/val/test splits are allocated here.

The pipeline this notebook follows is as follows:


```text
per-source manifests (data/interim/*.parquet)
        |
        v
combine_manifests()          -- into one DataFrame standardized to canonical columns
        |
        v
drop is_corrupt rows          -- ignored going forward
        |
        v
assign_group_ids()             -- cross-source near-duplicate clustering (banded LSH)
        |
        v
apply_matching()                -- bias-match real GenImage images (size + JPEG QF)
        |
        v
assign_full_splits()           -- group-aware GenImage split + fixed lookup for
                                   coco / ntire / raise
        |
        v
assert_no_leakage()             -- hard stop if any group/hash spans two splits
        |
        v
save data/interim/manifest_final.parquet
        |
        v
build_processed_dataset()      -- resize + re-encode every surviving image
        |
        v
data/processed/<split>/<ai|human>/<sha256>.jpg
        +
data/processed/manifest.parquet

### 0. Setup

In [2]:
%load_ext autoreload
%autoreload 2

## Standard Libraries
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

## Project module imports
from ai_detector.data.selection import(
    SubsetConfig, combine_manifests, assign_group_ids, apply_matching, assign_full_splits, 
    normalize_columns, shortcut_probe)
from ai_detector.data.manifest import(assert_no_leakage, LeakageError)
from ai_detector.data import viz
from ai_detector.preprocessing.image_ops import PreprocessConfig, build_processed_dataset
from ai_detector.data.integrity import near_duplicate_pairs

## Path setup
PROJECT_ROOT = Path.cwd().parent
DATA = PROJECT_ROOT / "data"
INTERIM = DATA / "interim"
PROCESSED = DATA / "processed"
RAW = DATA / "raw"
CONFIG_PATH = PROJECT_ROOT / "configs" / "data" / "subset_v1.yaml"

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

## Preprocessing and data splitting will draw from a yaml file in configs/data
cfg = SubsetConfig.from_yaml(CONFIG_PATH)
raw_cfg = yaml.safe_load(CONFIG_PATH.read_text())
preprocess_cfg = PreprocessConfig.from_yaml_dict(raw_cfg.get("preprocessing", {}))

print(cfg)
print(preprocess_cfg)

SubsetConfig(name='subset_v1', seed=42, real_generator_token='nature', train_generators=['stable_diffusion_v_1_4', 'stable_diffusion_v_1_5', 'glide', 'adm', 'vqdm'], ood_generators=['midjourney', 'wukong', 'biggan'], min_side=450, max_side=550, jpeg_qf=98, jpeg_qf_tolerance=2, val_fraction=0.1, test_in_dist_fraction=0.1, pilot_n_per_stratum=60)
PreprocessConfig(image_size=512, jpeg_qf=98, resample=<Resampling.LANCZOS: 1>)


### 1. Load and combine per-source manifests
Draws from the per-source manifests stored in `data/interim`

In [3]:
manifest_paths = sorted(INTERIM.glob("manifest_*.parquet"))
print(f"Found {len(manifest_paths)} manifest files:")
for p in manifest_paths:
    print(" -", p.name)

df = combine_manifests(manifest_paths)
print(f"\nCombined manifest: {len(df):,} rows, {df['source'].nunique()} sources")
display(df.groupby(["source", "label"]).size().unstack(fill_value=0))

Found 4 manifest files:
 - manifest_coco.parquet
 - manifest_genimage_tiny.parquet
 - manifest_ntire.parquet
 - manifest_raise.parquet

Combined manifest: 90,300 rows, 4 sources


label,0,1
source,,
coco,5000,0
genimage,17500,17500
ntire,17982,32018
raise,300,0


The combined manifests is sampled below:

In [4]:
display(df.sample(7))
print(df["is_corrupt"].value_counts())

,path,source,sha256,phash,label,generator,split,content_class,group_id,width,height,file_format,file_size_bytes,jpeg_qf,mode,is_corrupt,error
75248,ntire/shard_0/images/b3a9b0a63f9ac5bcce85.jpg,ntire,ffe6699c3320d88d725f2a5be2fa3611e91792da103446...,6a2c9d4b8a52c5d3,1,mixed,test_wild,None,None,768,1024,JPEG,134475,95,RGB,False,None
33762,tiny_genimage/imagenet_glide/train/nature/n044...,genimage,86275533a93670ddfd56d4ba7c246bfbcae55ed47d719e...,66f1968b9c82fd10,0,imagenet_glide,train,None,None,331,500,JPEG,100223,96.0,RGB,False,None
84992,ntire/shard_0/images/e64f1f3caf08586c51ec.jpg,ntire,573067fc7a776cafcaf77371a5bb09dbfd2fad4a46c892...,7b58584be52c5525,0,mixed,test_wild,None,None,2416,3216,JPEG,2353938,95,RGB,False,None
35353,tiny_genimage/imagenet_midjourney/train/ai/249...,genimage,dccda51daa8dda42ef1bb763e4e61e137ff3f4616beeef...,2b3cd4c32a751b1c,1,imagenet_midjourney,train,None,None,1024,1024,PNG,1477168,NaN,RGB,False,None
50198,ntire/shard_0/images/346db021928c92725e98.jpg,ntire,5854e883b199c24284ed37f6ebbffa371ed1cfcef03acb...,561b1b931edac069,0,mixed,test_wild,None,None,3024,4032,JPEG,2849783,95,RGB,False,None
83174,ntire/shard_0/images/dca3ce536386114f44e5.jpg,ntire,6f66b938d6b5acba94630deb553f1e5f6dc6319ee66ba8...,269fd1b853a54c2c,1,mixed,test_wild,None,None,512,400,JPEG,49512,95,RGB,False,None
15851,tiny_genimage/imagenet_ai_0424_sdv5/train/ai/4...,genimage,8e86e936833465c90f5d2867326b120e5e76d4fbf537ee...,5a9055199bd938b9,1,imagenet_ai_0424_sdv5,train,None,None,512,512,PNG,439161,NaN,RGB,False,None


is_corrupt
False    90300
Name: count, dtype: int64


### 2. Drop corrupt image rows
In the event corrupt images are present. These are flagged by `probe_decodability()` when the individual source-specific manifests were created. They lack a usable `phash` and therefore cannot be preprocessed/grouped/split, therefore they must be removed.

In [5]:
n_before= len(df)
corrupt= df[df["is_corrupt"]]

if len(corrupt):
    display(corrupt[["path", "source", "error"]].head(10))
df = df[~df["is_corrupt"]].reset_index(drop=True)
print(f"Dropped {n_before - len(df)} corrupt rows ({(n_before - len(df)) / n_before:.2%})")
print(f"Remaining: {len(df):,} rows")

Dropped 0 corrupt rows (0.00%)
Remaining: 90,300 rows


### 3. Cross-source Near-duplicate clustering
We initially implemented a Banded LSH + UnionFind (`integrity.py`) pipeline to group similar images within a particular source. This time, we apply the same grouping strategy across the entire combined manifest. 

Rather check for similar images on a pairwise basis that may result in potentially billions of image pair combinations, the Banded LSH approach breaks these potential comparisons into a smaller candidate set

In [13]:
## Examining the phash distributions 
# Which images would have the exact same phash?
df['phash'].value_counts()


phash
0000000000000000    29
12926d6d92926d6d     2
63ba9d86612d84b5     2
42ca4e4873515ddd     2
23dc954bdb1c2561     2
                    ..
2d5c5232cdcd34aa     1
4849f3f13c8ec703     1
1285ccd27d7832b5     1
50c3d38bc6c8a2fa     1
78cd4e39893b0fa0     1
Name: count, Length: 90267, dtype: int64

In [11]:
dedup_cfg= raw_cfg.get("dedup", {}) # Get grouping parameters from yaml file
df= assign_group_ids(df, **dedup_cfg)
    # max_distance and n_bands are unpacked from the yaml config to kwargs
    # If an error is thrown here, ensure paramater values have been added to the yaml file

group_sizes= df.groupby("group_id").size()
print(f"{df["group_id"].nunique():,} distinct groups from {len(df)} images")
print(f"Largest cluster size: {group_sizes.max()}")
display(group_sizes.value_counts().sort_index().head(10))

90,253 distinct groups from 90300 images
Largest cluster size: 29


1     90235
2        15
3         2
29        1
Name: count, dtype: int64

Low duplicate rate is expected as the images are from independent sources. This is good. The cluster of 29 images is however suspicious. These probably indicate near-uniform images (e.g.- flat sky, solid black/white frames, etc.) We will investigate them visually and drop those that provide no visual distinctions.